# 02 · Clean the data and build the analysis tables

This notebook turns the six raw CSVs into two clean tables for analysis. The raw files are not changed.

Cleaning rules, based on the checks in `01_data_checks.ipynb`:

1. **Drop exact duplicate rows.** About 3% of rows repeat the same skill for the same vacancy. They add no information.
2. **Standardise canton.** Missing values and "Non précisé" both become "Unknown".
3. **Build a reliable vacancy identifier.** `vacancy_id` is sometimes reused for different vacancies, both within a file and across files. A vacancy is identified by `source_file`, `vacancy_id`, `year`, `month`, `occupation_code`, `nace_code`, `canton` and `positions` together.
4. **Mark the incomplete months.** 2026 is complete up to August only. September 2026 onwards is flagged.

Output, saved to `data/processed/`:

- **vacancies**: one row per vacancy
- **vacancy_skills**: one row per vacancy–skill pair

Loading and processing can take a few minutes.

## 0. Settings and loading

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.api.types import union_categoricals
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Support running from either the project root or notebooks/.
project_dir = Path.cwd()
if not (project_dir / 'data' / 'raw').is_dir():
    project_dir = project_dir.parent

raw_dir = project_dir / 'data' / 'raw'
processed_dir = project_dir / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

files = sorted(raw_dir.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'No CSV files found in {raw_dir.resolve()}')

# Last month with complete data (from notebook 01: September 2026 has very few vacancies).
LAST_COMPLETE_YEAR = 2026
LAST_COMPLETE_MONTH = 8

TEXT_COLS = ['skill', 'skill_uri', 'canton', 'occupation_code',
             'occupation_label', 'nace_code', 'nace_label']
DTYPES = {'vacancy_id': 'int64', 'positions': 'int32',
          'month': 'int8', 'year': 'int16'}
DTYPES.update({col: 'category' for col in TEXT_COLS})
ORIGINAL_COLS = ['vacancy_id', 'skill', 'skill_uri', 'positions', 'month', 'year',
                 'canton', 'occupation_code', 'occupation_label', 'nace_code', 'nace_label']

frames = []
for file in files:
    part = pd.read_csv(file, dtype=DTYPES, usecols=ORIGINAL_COLS)
    part['source_file'] = file.stem.replace('datasc-skills-vacancies-', '')
    frames.append(part)
    print(f'{file.name}: {len(part):,} rows')

# Give every text column the same categories in all files so they combine cleanly.
for col in TEXT_COLS:
    cats = union_categoricals([f[col] for f in frames]).categories
    for f in frames:
        f[col] = f[col].cat.set_categories(cats)

df = pd.concat(frames, ignore_index=True)
df['source_file'] = df['source_file'].astype('category')
del frames

# Keep track of how many rows each step removes.
steps = [('Raw rows', len(df))]
print(f'\nTotal: {len(df):,} rows')

## 1. Drop exact duplicate rows

A row is a duplicate if all 11 original columns are identical to another row in the same file.

In [ ]:
is_dup = df.duplicated(subset=['source_file'] + ORIGINAL_COLS)
print('Duplicate rows removed per file:')
display(is_dup.groupby(df['source_file'], observed=True).sum())

df = df[~is_dup].reset_index(drop=True)
steps.append(('After dropping exact duplicates', len(df)))
print(f'Rows removed: {is_dup.sum():,}. Rows left: {len(df):,}')

## 2. Standardise canton

Missing canton and "Non précisé" (not specified) mean the same thing, so both become "Unknown". This is done before building the vacancy identifier, so the two versions don't split one vacancy in two.

Missing sector (`nace_code`) is left as missing. It only affects about 1% of rows.

In [ ]:
print('Canton values before:')
display(df['canton'].value_counts(dropna=False))

canton = df['canton'].astype('object').where(df['canton'].notna(), 'Unknown')
canton = canton.replace({'Non précisé': 'Unknown'})
df['canton'] = canton.astype('category')

print('Canton values after:')
display(df['canton'].value_counts(dropna=False))

## 3. Build the vacancy identifier

`vacancy_key` is a new number that is unique for each combination of the columns below. The original `vacancy_id` is kept for reference.

In [ ]:
VACANCY_COLS = ['source_file', 'vacancy_id', 'year', 'month',
                'occupation_code', 'nace_code', 'canton', 'positions']

df['vacancy_key'] = (df.groupby(VACANCY_COLS, observed=True, dropna=False, sort=True)
                       .ngroup().astype('int64'))

n_ids = df.groupby(['source_file', 'vacancy_id'], observed=True).ngroups
n_keys = df['vacancy_key'].nunique()
print(f'(file, vacancy_id) combinations: {n_ids:,}')
print(f'Vacancies with the new key:      {n_keys:,}')
print(f'Extra vacancies found by splitting reused IDs: {n_keys - n_ids:,}')

keys_per_id = df.groupby(['source_file', 'vacancy_id'], observed=True)['vacancy_key'].nunique()
print('\nHow many vacancies each (file, vacancy_id) splits into:')
display(keys_per_id.value_counts().sort_index().rename('number_of_ids'))

In [ ]:
# After splitting, a skill should appear at most once per vacancy.
repeated = df['skill_uri'].notna() & df.duplicated(subset=['vacancy_key', 'skill_uri'])
print(f'Skills still repeated within a vacancy: {repeated.sum():,}')

if repeated.any():
    # Same skill and same vacancy, differing only in the label text: keep one row.
    print('Example:')
    example_key = df.loc[repeated, 'vacancy_key'].iloc[0]
    display(df[df['vacancy_key'] == example_key].head(15))
    df = df[~repeated].reset_index(drop=True)
    print(f'Removed {repeated.sum():,} repeated skill rows.')

steps.append(('After removing repeated skills', len(df)))

## 4. Mark incomplete months

`in_complete_period` is `True` for every month up to and including August 2026, and `False` after it. Use it to filter before comparing years.

In [ ]:
period = df['year'].astype('int32') * 100 + df['month'].astype('int32')
df['in_complete_period'] = period <= LAST_COMPLETE_YEAR * 100 + LAST_COMPLETE_MONTH

print('Rows outside the complete period, by year and month:')
display(df.loc[~df['in_complete_period']].groupby(['year', 'month']).size())

## 5. Build the two tables

**vacancies**: one row per vacancy, with `n_skills` (number of skills listed) and `has_skills`.

**vacancy_skills**: one row per vacancy–skill pair. Vacancies with no skill have no rows here, but they stay in `vacancies`. Join the two tables on `vacancy_key` to get the vacancy details.

In [ ]:
VACANCY_TABLE_COLS = ['vacancy_key', 'vacancy_id', 'source_file', 'year', 'month',
                      'canton', 'occupation_code', 'occupation_label',
                      'nace_code', 'nace_label', 'positions', 'in_complete_period']

vacancies = (df.drop_duplicates('vacancy_key')[VACANCY_TABLE_COLS]
               .sort_values('vacancy_key').reset_index(drop=True))

n_skills = df.groupby('vacancy_key')['skill_uri'].count()
vacancies['n_skills'] = vacancies['vacancy_key'].map(n_skills).astype('int32')
vacancies['has_skills'] = vacancies['n_skills'] > 0

# First day of the month, useful for time-series charts.
vacancies['date'] = pd.to_datetime(dict(year=vacancies['year'],
                                        month=vacancies['month'], day=1))

vacancy_skills = (df.loc[df['skill_uri'].notna(), ['vacancy_key', 'skill_uri', 'skill']]
                    .sort_values(['vacancy_key', 'skill'])
                    .reset_index(drop=True))
for col in ['skill_uri', 'skill']:
    vacancy_skills[col] = vacancy_skills[col].cat.remove_unused_categories()

print(f'vacancies:      {len(vacancies):,} rows')
print(f'vacancy_skills: {len(vacancy_skills):,} rows')
display(vacancies.head())
display(vacancy_skills.head())

## 6. Checks before saving

Each line should say `OK`.

In [ ]:
def check(name, condition):
    print(f'{"OK  " if condition else "FAIL"}  {name}')
    return condition

results = [
    check('vacancy_key is unique in vacancies', vacancies['vacancy_key'].is_unique),
    check('no repeated skill within a vacancy',
          not vacancy_skills.duplicated(['vacancy_key', 'skill_uri']).any()),
    check('every skill row belongs to a vacancy',
          vacancy_skills['vacancy_key'].isin(vacancies['vacancy_key']).all()),
    check('n_skills adds up to the skill table',
          vacancies['n_skills'].sum() == len(vacancy_skills)),
    check('positions are all at least 1', (vacancies['positions'] >= 1).all()),
    check('months are between 1 and 12', vacancies['month'].between(1, 12).all()),
    check('each skill URI has one label',
          (vacancy_skills.groupby('skill_uri', observed=True)['skill'].nunique() <= 1).all()),
]
if not all(results):
    print('\nSome checks failed. Look into them before using the tables.')

## 7. Save the tables

Parquet is used if `pyarrow` is installed (faster and smaller). Otherwise the tables are saved as compressed CSV.

In [ ]:
try:
    import pyarrow  # noqa: F401
    vacancies.to_parquet(processed_dir / 'vacancies.parquet', index=False)
    vacancy_skills.to_parquet(processed_dir / 'vacancy_skills.parquet', index=False)
    saved_as = 'parquet'
except ImportError:
    vacancies.to_csv(processed_dir / 'vacancies.csv.gz', index=False)
    vacancy_skills.to_csv(processed_dir / 'vacancy_skills.csv.gz', index=False)
    saved_as = 'csv.gz'

print(f'Saved as {saved_as} in {processed_dir.resolve()}:')
for path in sorted(processed_dir.glob('vacanc*')):
    print(f'  {path.name}: {path.stat().st_size / 1e6:.1f} MB')

## 8. Before and after

In [ ]:
step_table = pd.DataFrame(steps, columns=['step', 'rows'])
step_table['rows_removed'] = (-step_table['rows'].diff()).fillna(0).astype(int)
display(step_table)

complete = vacancies[vacancies['in_complete_period']]
with_skills = complete[complete['has_skills']]
by_year = pd.DataFrame({
    'vacancies': complete.groupby('year').size(),
    'posts': complete.groupby('year')['positions'].sum(),
    'vacancies_with_skills': with_skills.groupby('year').size(),
    'avg_skills_per_vacancy': with_skills.groupby('year')['n_skills'].mean().round(2),
})
by_year['pct_with_skills'] = (by_year['vacancies_with_skills'] / by_year['vacancies'] * 100).round(1)
print('Clean data by year (complete months only; 2026 = January to August):')
display(by_year)

## What to note down

- How many rows each step removed
- How many extra vacancies were found by splitting reused IDs
- Whether every check printed `OK`
- The clean vacancies per year and the share with skills

The next notebook will load the two saved tables instead of the raw CSVs.